## Description
This notebook runs the model for specified scenarios

### Imports

In [8]:
import pandas as pd
import numpy as np
import pylab as plt

import pypsa

from pathlib import Path
import gc
from itertools import product

# Import the public function from the package you completed
from energy_model import model_run

### Setting model parameters

In [9]:
# Default parameters

# model years
years = [2025, 2030, 2040, 2050]

# intra-year temporal resolution
t_res = 15 # hours

# discount rates
rt = 0.07 # technical discount rate
rs = 0.01 # social discount rate

# passenger cars conversion factor
passenger_cars_cf = 40*0.1  # km/h

# Define where the files are searched from
db_dir = "input_databases/default"

### Plotting functions

In [10]:
# Handy plotting functions. 
# Note that these are good to quickly diagnose your runs, 
# but for your final presentation and paper you will need
# nicer-looking plots, following the data-visualization 
# guidelines we discussed in class!

def plot_main_ebs(scen):
    fig, ax = plt.subplots(1, 1)
    fig.set_size_inches(10, 4)

    df = scen.statistics.energy_balance(bus_carrier='Hydrogen')
    df = df.set_index(df.index.get_level_values(1)).T
    df.plot(kind='bar', stacked='True', rot=0, colormap="tab20", ax=ax)
    ax.set_ylabel('MWh')
    ax.set_title(scen.name)

    fig.tight_layout()

def plot_main_ebs_complete(scen):
    fig, ax = plt.subplots(1,5)
    fig.set_size_inches(20,4)

    for n,c in enumerate(['Residential electricity',
            'Residential heat',
            'Transportation',
            'Industrial machinery',
            'Industrial HTP']): 
        df = scen.statistics.energy_balance(bus_carrier=c)
        df = df.set_index(df.index.get_level_values(1)).T
        df.plot(kind='bar',stacked='True', rot=0, colormap="tab20",ax=ax[n])
        if c=='Transportation':
            ax[n].set_ylabel('Mvkm')
        else:
            ax[n].set_ylabel('MWh')
        ax[n].set_title(c)

    fig.tight_layout()

def plot_elc_emi(scen):
    fig, ax = plt.subplots(1,5)
    fig.set_size_inches(20,4)

    for n,c in enumerate(['Electricity','CO2 atmosphere', 'CO2 storage']):
        df = scen.statistics.energy_balance(bus_carrier=c)
        df = df.set_index(df.index.get_level_values(1)).T
        df.plot(kind='bar',stacked='True', rot=0, colormap="tab20", ax=ax[n])

        if 'CO2' in c:
            ax[n].set_ylabel('tCO2')
        else:
            ax[n].set_ylabel('MWh')
        ax[n].set_title(c)
        ax[n].legend(bbox_to_anchor=(1,-0.5))

    # Let's add cumulative CO2
    for n,c in enumerate(['CO2 atmosphere', 'CO2 storage']):
        df = scen.statistics.energy_balance(bus_carrier=c).cumsum(axis=1)
        df = df.set_index(df.index.get_level_values(1))
        df.T.plot(kind='bar', stacked=True, colormap='tab20', ax=ax[n+3])
        ax[n+3].set_ylabel('tCO2')
        ax[n+3].set_title(f'Cumulative {c}')
        ax[n+3].legend(bbox_to_anchor=(1,-0.5))

    fig.tight_layout()


### Running functions

In [11]:
def run_scenario(out_dir: Path = Path("results"), gas: float = 1, elec: float = 1, co2: float | None = None) -> None:
    """Solve one scenario and write it to disk; skip if it already exists."""
    name = f"gas_x{gas}_elec_x{elec}"
    if co2 is not None:
        name += f"_co2_{co2}"
    out_file = out_dir / f"{name}.nc"

    if out_file.exists():
        print(f"{name} already done, skipping")
        return
    print(f"Running {name}...")

    n = model_run(
        db_dir, years, t_res, passenger_cars_cf, rt, rs,
        gas_multiplier=gas,
        electrolysis_multiplier=elec,
        carbon_price=co2,
    )
    n.name = name
    n.export_to_netcdf(out_file)

def save_run(out_dir: Path = Path("results"), gas: float = 1, elec: float = 1, co2: float | None = None) -> None:
    """Run one scenario; print and continue if it fails."""
    try:
        run_scenario(out_dir, gas, elec, co2)
    except Exception as e:
        print(f"Gas: {gas}, Elec: {elec}, co2: {co2} failed: {e}")
    finally:
        gc.collect()

### Setting scenario parameters and running the model

In [12]:
# Define ranges for gas and electrolysis multipliers to model the various
# gas shock and electrolysis tech investment reduction scenarios 
gas_multipliers = [1, 3, 5.5, 8]
electrolysis_multipliers = [1, 0.75, 0.5, 0.25]
final_results = Path("results/final_results")

for gas, elec in product(gas_multipliers, electrolysis_multipliers):
    save_run(out_dir=final_results, gas=gas, elec=elec)

gas_x1_elec_x1 already done, skipping
gas_x1_elec_x0.75 already done, skipping
gas_x1_elec_x0.5 already done, skipping
gas_x1_elec_x0.25 already done, skipping
gas_x3_elec_x1 already done, skipping
gas_x3_elec_x0.75 already done, skipping
gas_x3_elec_x0.5 already done, skipping
gas_x3_elec_x0.25 already done, skipping
gas_x5.5_elec_x1 already done, skipping
gas_x5.5_elec_x0.75 already done, skipping
gas_x5.5_elec_x0.5 already done, skipping
gas_x5.5_elec_x0.25 already done, skipping
gas_x8_elec_x1 already done, skipping
gas_x8_elec_x0.75 already done, skipping
gas_x8_elec_x0.5 already done, skipping
gas_x8_elec_x0.25 already done, skipping


### Setting carbon prices for carbon price sensitivity testing

In [ ]:
carbon_prices = [0] + list(range(100, 325, 25)) + list(range(100, 105, 1))
carbon_dir = Path('results/carbon_runs_final')

for price in carbon_prices:
    save_run(out_dir=carbon_dir, co2 = price)

Running gas_x1_elec_x1_co2_0...
2025
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 222.02it/s]

Set parameter Username
Set parameter LicenseID to value 2858750
Academic license - for non-commercial use only - expires 2027-08-31


Read LP format model from file C:\Users\rosam\AppData\Local\Temp\linopy-problem-u560428q.lp
Reading time = 0.61 seconds
obj: 324228 rows, 151353 columns, 767316 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2325588026581.538
Running gas_x1_elec_x1_co2_100...
2025
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 256.81it/s]


Set parameter Username
Set parameter LicenseID to value 2858750
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file C:\Users\rosam\AppData\Local\Temp\linopy-problem-xq5xudts.lp
Reading time = 0.51 seconds
obj: 324228 rows, 151353 columns, 767316 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2510644019782.025
Running gas_x1_elec_x1_co2_101...
2025
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 223.06it/s]

Set parameter Username
Set parameter LicenseID to value 2858750
Academic license - for non-commercial use only - expires 2027-08-31


Read LP format model from file C:\Users\rosam\AppData\Local\Temp\linopy-problem-at5v7_ge.lp
Reading time = 0.55 seconds
obj: 324228 rows, 151353 columns, 767316 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2512405688036.3457
Running gas_x1_elec_x1_co2_102...
2025
2030
2040
2050


### Running carbon price sensitivity models

In [ ]:
# Couple of tests for the gas prices

gas_multipliers = [1, 1.5, 1.8, 1.9, 2, 2.5, 3]
gas_out_dir = Path('results/gas_tests')

for gas in gas_multipliers:
    save_run(out_dir = gas_out_dir, gas=gas)

Running gas_x1.0_elec_x1...
2025
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 305.85it/s]


Set parameter Username
Set parameter LicenseID to value 2858750
Academic license - for non-commercial use only - expires 2027-08-31
Read LP format model from file C:\Users\rosam\AppData\Local\Temp\linopy-problem-v6h88fkg.lp
Reading time = 0.20 seconds
obj: 139413 rows, 65106 columns, 329916 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2447944272030.493
Running gas_x1.0_elec_x0.75...
2025
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 244.35it/s]

Set parameter Username
Set parameter LicenseID to value 2858750
Academic license - for non-commercial use only - expires 2027-08-31


...

Set parameter Username
Set parameter LicenseID to value 2858750
Academic license - for non-commercial use only - expires 2027-08-31


Read LP format model from file C:\Users\rosam\AppData\Local\Temp\linopy-problem-fjhr6aj5.lp
Reading time = 0.28 seconds
obj: 139413 rows, 65106 columns, 329916 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2995722846330.471
Running gas_x3.0_elec_x0.25...
2025
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 198.95it/s]

Set parameter Username
Set parameter LicenseID to value 2858750
Academic license - for non-commercial use only - expires 2027-08-31


Read LP format model from file C:\Users\rosam\AppData\Local\Temp\linopy-problem-796x2f5p.lp
Reading time = 0.31 seconds
obj: 139413 rows, 65106 columns, 329916 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2994328476044.442


# No hydrogen baseline

In [8]:

no_hydro_db = "input_databases/default_no_hydrogen"

n_no_hydro = model_run(
    no_hydro_db,
    years,
    t_res,
    passenger_cars_cf,
    rt,
    rs,
    gas_multiplier=1,
    electrolysis_multiplier=1,
    carbon_price=None
)

assert "Hydrogen" not in n_no_hydro.buses.index, \
    "ERROR: Hydrogen bus is still present!"

assert "industrial_hydrogen" not in n_no_hydro.loads.index, \
    "ERROR: Hydrogen demand is still present!"

print("No-hydrogen model verified successfully.")
print("Buses:", len(n_no_hydro.buses))
print("Links:", len(n_no_hydro.links))
print("Snapshots:", len(n_no_hydro.snapshots))

out_dir = Path("results/no_hydro")
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "gas_x1_elec_x1_no_hydro.nc"

n_no_hydro.name = "baseline_no_hydrogen"
n_no_hydro.export_to_netcdf(out_file)

print(f"Saved no-hydrogen baseline to: {out_file}")


2025
2030
2040
2050


Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 496.03it/s]

Set parameter Username
Set parameter LicenseID to value 2858748
Academic license - for non-commercial use only - expires 2027-08-31


Read LP format model from file /private/var/folders/_s/3m5r3tk90cb5vvjhzy5smf0h0000gn/T/linopy-problem-n16ow17z.lp
Reading time = 0.17 seconds
obj: 121825 rows, 56808 columns, 280200 nonzeros
Set parameter LogFile to value "results//solver_run.log"
Set parameter LogToConsole to value 0


Model Status: ok
Termination Condition: optimal
Objective value: 2396511979795.629
No-hydrogen model verified successfully.
Buses: 12
Links: 66
Snapshots: 1004
Saved no-hydrogen baseline to: results/no_hydro/gas_x1_elec_x1_no_hydro.nc
